In [ ]:
# import json
# import ijson

# with open(r""path/to/my/file", "r") as infile, open("public_v2.ndjson", "w") as outfile:
#     for key, value in ijson.kvitems(infile, ''):
#         value["log_id"] = key
#         outfile.write(json.dumps(value) + "\n")

In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect()

# Limitiamo a 10.000 righe per velocità
df = con.execute("""
    SELECT *
    FROM read_json_auto('public_v2.ndjson')
    LIMIT 1000
""").df()

df.info()

In [ ]:
# %% [2] - Aggiungiamo pochi attacchi artificiali
attacks = [
    {
        "referrer": "http://search.lib.auth.gr/Search/<script>alert('XSS')</script>",
        "request": "search.lib.auth.gr:80 185.22.12.44 - - [01/Mar/2018:00:00:15 +0200] GET /Search?q=<script>alert('XSS')</script> HTTP/1.1 400 1200 http://search.lib.auth.gr/Search/<script>alert('XSS')</script> Mozilla/5.0",
        "method": "GET",
        "resource": "/Search?q=<script>alert('XSS')</script>",
        "bytes": "1200",
        "response": "400",
        "ip": "185.22.12.44",
        "useragent": "Mozilla/5.0",
        "timestamp": "2018-02-28T22:00:15.000Z",
        "log_id": "attack_xss"
    },
    {
        "referrer": "http://evil.com",
        "request": "search.lib.auth.gr:80 45.77.88.12 - - [01/Mar/2018:00:00:20 +0200] GET /login?user=admin' OR 1=1-- HTTP/1.1 500 800 http://evil.com Mozilla/5.0",
        "method": "GET",
        "resource": "/login?user=admin' OR 1=1--",
        "bytes": "800",
        "response": "500",
        "ip": "45.77.88.12",
        "useragent": "Mozilla/5.0",
        "timestamp": "2018-02-28T22:00:20.000Z",
        "log_id": "attack_sql"
    },
    {
        "referrer": "-",
        "request": "search.lib.auth.gr:80 91.200.12.77 - - [01/Mar/2018:00:00:25 +0200] GET /../../../../etc/passwd HTTP/1.1 403 300 - curl/7.58.0",
        "method": "GET",
        "resource": "/../../../../etc/passwd",
        "bytes": "300",
        "response": "403",
        "ip": "91.200.12.77",
        "useragent": "curl/7.58.0",
        "timestamp": "2018-02-28T22:00:25.000Z",
        "log_id": "attack_traversal"
    },
    {
        "referrer": "-",
        "request": "search.lib.auth.gr:80 203.0.113.45 - - [01/Mar/2018:00:00:30 +0200] GET /wp-admin/install.php HTTP/1.1 404 250 - sqlmap/1.7",
        "method": "GET",
        "resource": "/wp-admin/install.php",
        "bytes": "250",
        "response": "404",
        "ip": "203.0.113.45",
        "useragent": "sqlmap/1.7",
        "timestamp": "2018-02-28T22:00:30.000Z",
        "log_id": "attack_scanner"
    }
]

df = pd.concat([df, pd.DataFrame(attacks)], ignore_index=True)

df["timestamp"] = pd.to_datetime(df["timestamp"])


In [ ]:
#lughezza della resource richiesta

df["resource_len"] = df["resource"].str.len()
df.shape

In [ ]:
#da valutare se devo fare un primo controllo anche io sui quartili
from detect import has_suspicious_request_len

df = has_suspicious_request_len(df)

df.shape

In [ ]:
#troppi special characters
df["special_chars"] = df["request"].str.count(r"[<>\'\"%;(){}]*+@")
df["special_chars"].value_counts()

In [ ]:
#directory traversale
df["has_traversal"] = df["request"].str.contains(r"\.\./", regex=True, na=False).astype(int)
df["has_traversal"].value_counts()

In [ ]:
# sql injection

df["has_sql_keywords"] = df["request"].str.contains(
    r"union|select|insert|drop|or 1=1|--|admin",
    case=False,
    na=False
).astype(int)

df["has_sql_keywords"].value_counts()

In [ ]:
#script injection
df["has_script_tag"] = df["request"].str.contains(
    r"<script>",
    case=False,
    na=False
).astype(int)
df["has_script_tag"].value_counts()

In [ ]:
#possibile scanner da parte di bot
df["scanner_agent"] = df["useragent"].str.contains(
    r"sqlmap|nikto|curl|nmap|dirbuster",
    case=False,
    na=False
).astype(int)
df["useragent"].value_counts()

In [ ]:
# status code di errore
df["is_error"] = (df["response"].astype(int) >= 400).astype(int)

df['is_error'].value_counts()

In [ ]:
# numero di query eccessivo
df["num_params"] = df["resource"].str.count("=")
df["num_params"].value_counts()

In [ ]:
# numero di richieste per IP
df["requests_per_ip"] = df.groupby("ip")["ip"].transform("count")


In [ ]:
from detect import entropy

df["request_entropy"] = df["request"].apply(lambda x: entropy(str(x)))


In [ ]:
df['bytes_num'] = pd.to_numeric(df['bytes'], errors='coerce').fillna(0)
resource_avg_bytes = df.groupby('resource')['bytes_num'].transform('mean')

df['bytes_ratio'] = df['bytes_num'] / (resource_avg_bytes + 1)
df['bytes_ratio'].value_counts()

In [ ]:
df['is_image'] = df['resource'].str.contains(r'\.(jpg|jpeg|png|gif|svg)', case=False, na=False).astype(int)

In [ ]:
bot_keywords = [
    'bot', 'crawler', 'spider', 'slurp', 'bingpreview', 'thumbshot',
    'bubing', 'dotbot', 'icc-crawler', 'googlebot', 'yandex', 'baidu'
]
bot_pattern = '|'.join(bot_keywords)
df["is_known_bot"] = df["useragent"].str.contains(bot_pattern, case=False, na=False).astype(int)

DA QUI IN POI ABBIAMO L'ALGORITMO DI ISOLATION FOREST
1. L'algoritmo non accetta stringhe o date. Dobbiamo isolare solo le colonne numeriche (quelle che hai creato con tanta cura).


In [ ]:
from sklearn.ensemble import IsolationForest

# Definiamo le colonne che l'algoritmo deve "studiare"
features = [
    'resource_len', 'has_suspicious_request_len', 'special_chars',
    'has_traversal', 'has_sql_keywords', 'has_script_tag',
    'scanner_agent', 'is_error', 'num_params',
    'requests_per_ip', 'request_entropy', 'bytes_num',
    'bytes_ratio', 'is_image'
]

X = df[features]
df['is_attack_real'] = df['log_id'].str.contains('attack').astype(int)
df.loc[df['is_attack_real']==1, 'is_attack_real']=-1
df.loc[df['is_attack_real'] == 0, 'is_attack_real'] =1

2. Configurazione e Addestramento
L'Isolation Forest funziona creando alberi decisionali casuali: le anomalie vengono "isolate" più velocemente (richiedono meno biforcazioni).


In [ ]:
# contamination è la % stimata di attacchi nel dataset (es. 0.1% = 0.001)
from sklearn.metrics import roc_auc_score, make_scorer
from sklearn.preprocessing import MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

numeric_cols = ['resource_len', 'special_chars','num_params','requests_per_ip', 'request_entropy', 'bytes_num','bytes_ratio']
model = IsolationForest(n_estimators=100, contamination=0.01, random_state=42)
transformer = ColumnTransformer([
    ('numeric_cols', MinMaxScaler((0,1)), numeric_cols)
])
prod_pipe = Pipeline([
    ('transformer',transformer),
    ('model', model)
])

params_grid = {
    "model__n_estimators": [100, 500, 1000, 1500, 2000],
    "model__contamination":[0.01,0.015,0.020,0.025,0.03]
}

score_grid = {'rec': 'recall_macro',  # vero positivo/(vero positivo+falso negativo)
              'pres': 'precision_macro'}  # vero positivo/(vero positivo+falso positivo)
prod_val = GridSearchCV(estimator=prod_pipe, cv=3, param_grid= params_grid,scoring= score_grid, refit='rec', n_jobs=-1, error_score=True).fit(X,df['is_attack_real'])
best_model = prod_val.best_estimator_
best_results = pd.DataFrame(prod_val.cv_results_)
print(f'Configurazione ottimale: {prod_val.best_params_}\nCV Recall: {prod_val.best_score_}')

# Addestramento

3. Interpretazione dei risultati
Oltre al flag -1/1, è utilissimo guardare il decision_function, che dà un punteggio continuo: più è basso, più la richiesta è "strana".


In [ ]:
#Addesdtramento

df['anomaly_score'] = best_model.fit_predict(X)

#Punteggio grezzo: valori negativi = più anomalo
df['scores'] = best_model.decision_function(X)

# Vediamo se i tuoi attacchi artificiali sono stati scovati
print(df[df['log_id'].str.contains('attack')][['log_id', 'anomaly_score', 'scores']])

4. Verifica dei Falsi Positivi
Ordina il dataset per il punteggio più basso per vedere cosa l'algoritmo considera "peggiore" tra i dati reali:

In [ ]:
# # Mostra le 10 richieste più sospette del dataset
# top_anomalies = df.sort_values(by='scores').head(10)
# print(top_anomalies[['request', 'scores']])

Dato che ho inserito degli attacchi manualmente (attack_xss, attack_sql, ecc.), posso calcolare una Matrice di Confusione semplificata per dimostrare quanto l'algoritmo è efficace nel trovare "l'ago nel pagliaio" che ho nascosto io.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report


# 1. Creiamo una colonna "realtà" (y_true)
# Assumiamo che tutto sia normale (0) tranne i log_id che iniziano con 'attack' (1)
df['is_attack_real'] = df['log_id'].str.contains('attack').astype(int)

# 2. Trasformiamo la predizione dell'Isolation Forest (y_pred)
# L'algoritmo sputa -1 per anomalia e 1 per normale. Lo convertiamo in 1 e 0.
df['is_attack_predicted'] = df['anomaly_score'].apply(lambda x: 1 if x == -1 else 0)

# 3. Generiamo la matrice
cm = confusion_matrix(df['is_attack_real'], df['is_attack_predicted'])

# 4. Visualizziamola graficamente
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normale', 'Attacco'],
            yticklabels=['Normale', 'Attacco'])
plt.xlabel('Predizione Modello')
plt.ylabel('Realtà (Ground Truth)')
plt.title('Matrice di Confusione')
plt.show()

# 5. Report di precisione e recall
print(classification_report(df['is_attack_real'], df['is_attack_predicted']))